# 10 Frozen Equilibrium Shift Diagnostics

Compare frozen equilibrium parameters with trailing realized spreads. These are diagnostics, not a rolling-recalibration strategy or a causal explanation of drawdown.

Run cells from top to bottom, or choose **Run All** for this notebook only. Each module saves its outputs for the next notebook. Restart the kernel after pulling code changes.


## 1. Imports and saved run


In [ ]:
%matplotlib inline
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src' / 'research_config.py').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Open Jupyter inside the Pairs_trading repository.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from src.project_io import OUTPUT_DIR, initialize, load_config, load_frame, save_frame, save_json

cfg = load_config()


## 2. Reconstruct frozen spreads

Use the same alpha, beta and equilibrium as the saved backtest.


In [ ]:
from src.backtest import compute_log_spread
from statsmodels.tsa.stattools import adfuller
train = load_frame('train_prices')
test = load_frame('test_prices')
full = pd.concat([train, test])
params = load_frame('pair_parameters')
rows = []
rolling_spreads = {}
for row in params.itertuples():
    spread = compute_log_spread(full, row.dependent, row.independent, row.alpha, row.beta)
    shift = (spread.rolling(63).mean() - row.mu) / np.sqrt(row.variance)
    rolling_spreads[row.pair] = shift.reindex(test.index)
    recent = spread.iloc[-252:]
    try:
        ordinary_adf_p = float(adfuller(recent, regression='c', autolag='AIC')[1])
    except ValueError:
        ordinary_adf_p = np.nan
    rows.append(dict(pair=row.pair, mean_absolute_shift_z=shift.reindex(test.index).abs().mean(),
        final_shift_z=shift.iloc[-1], trailing_ordinary_adf_p=ordinary_adf_p))
metrics = pd.DataFrame(rows).sort_values('mean_absolute_shift_z', ascending=False)
save_frame('equilibrium_shift_metrics', metrics)
save_frame('rolling_equilibrium_shift', pd.DataFrame(rolling_spreads))
display(metrics.head(15))


## 3. Visual inspection

Trailing ordinary ADF p-values here are residual diagnostics, not the formal formation Engle–Granger test. Large displacement is descriptive evidence and does not identify a unique drawdown mechanism.


In [ ]:
if not metrics.empty:
    examples = metrics.pair.head(4).tolist()
    pd.DataFrame(rolling_spreads)[examples].plot(figsize=(11, 4), title='Trailing 63-session mean displacement in frozen standard deviations')
    plt.axhline(0, color='black', linestyle='--')
    plt.show()


## Save module completion

Wait for this confirmation before moving to the next notebook.


In [ ]:
print(f'Completed. Files saved in {OUTPUT_DIR}')
